In [30]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, Polygon
import numpy as np
from pathlib import Path
import os
import sys

In [31]:
CURRENT_DIRECTORY =  os.getcwd()
PARENT_DIRECTORY = os.path.dirname(CURRENT_DIRECTORY)
print(PARENT_DIRECTORY)

BASE_DATASET_PATH = Path(PARENT_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


#### read the cleaned dataset containing OHCA incidences with their respective latitude and longtitude

In [11]:
dataset_filepath = str(BASE_DATASET_PATH / "3_OHCA_with_coords.xlsx")
ohca_df = pd.read_excel(dataset_filepath)
print(ohca_df.shape)

(27729, 16)


In [12]:
columns_to_drop = ["Unnamed: 0.2", "Unnamed: 0.1", "Unnamed: 0", "Location Type Other_x", "Location Type Other_y"]
ohca_df = ohca_df.drop(columns = columns_to_drop)
ohca_df.head(3)

,Case #,Site #,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,Time call received at dispatch center,Estimated time of arrest,lat,lon
0,SGSIN0213,2,2010-04-01,470146.0,NaN,Home Residence,HDB Level 7,23:29:17,23:35:00,1.334604,103.911889
1,SGSIN0218,2,2010-04-01,520926.0,NaN,Home Residence,HDB Level 2,14:18:54,14:05:00,1.346122,103.940989
2,SGSIN6480,6,2010-04-01,560565.0,NaN,Healthcare Facility,NKF Dialysis Centre,16:35:03,16:30:00,1.369882,103.858475


In [13]:
ohca_geo_df = gpd.GeoDataFrame(
    ohca_df,
    geometry = gpd.points_from_xy(
        ohca_df["lon"],
        ohca_df["lat"]
    ),
    crs = "EPSG:4326"
)

# reproject to 3414, use by Singapore
ohca_geo_df = ohca_geo_df.to_crs(3414)
ohca_geo_df.head(3)

# make a geometry column 
# gdf_scene_coordinates = gpd.GeoDataFrame(
#     df_scene_coordinates,
#     geometry = gpd.points_from_xy(
#         df_scene_coordinates["combined scene longitude"],
#         df_scene_coordinates["combined scene latitude"]        
#     ),
#     crs="EPSG:4326"  # WGS84 lat/lon
# )
# # project to 3375, used for malaysia (units: metres)
# gdf_scene_coordinates = gdf_scene_coordinates.to_crs("EPSG:3375")

,Case #,Site #,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,Time call received at dispatch center,Estimated time of arrest,lat,lon,geometry
0,SGSIN0213,2,2010-04-01,470146.0,NaN,Home Residence,HDB Level 7,23:29:17,23:35:00,1.334604,103.911889,POINT (36744.065 35199.375)
1,SGSIN0218,2,2010-04-01,520926.0,NaN,Home Residence,HDB Level 2,14:18:54,14:05:00,1.346122,103.940989,POINT (39982.538 36473.112)
2,SGSIN6480,6,2010-04-01,560565.0,NaN,Healthcare Facility,NKF Dialysis Centre,16:35:03,16:30:00,1.369882,103.858475,POINT (30799.605 39100.122)


In [16]:
# save to folder
output_filepath = str(BASE_DATASET_PATH / "3_OHCA_with_geometric_pts.csv")
ohca_geo_df.to_csv(output_filepath)

gpkg_output_filepath = str(BASE_DATASET_PATH / "3_OHCA_with_geometric_pts.gpkg")
ohca_geo_df.to_file(gpkg_output_filepath)

#### Adding 3_OHCA_with_geometric_pts.gpkg to sql for intersection with hectare grids

In [8]:
def construct_add_command(filename: str, database_name: str, username: str):

    cmd = [f"ogr2ogr -f PostgreSQL PG:\"dbname={database_name} user={username}\" ", 
           filename + ".gpkg",
           " -nln ", filename,
           " -lco GEOMETRY_NAME=geom ",
           "-lco FID=id ",
           "-nlt PROMOTE_TO_MULTI ",
           filename
    ]
    result = "".join(cmd)
    return result


def add_to_postgres(database_name: str, username: str, filename: str | None = None):
    """
    adds the layers from .gpkg file to postgreSQL
    Can't get PostgreSQL command to work,
    need to manually copy command output and run in terminal
    """

    cmd = construct_add_command(filename, database_name, username)
    print(cmd)

    cmd_indexes = f"""
CREATE INDEX IF NOT EXISTS hectare_grid_geom_idx ON hectare_grid USING GIST (geom);

CREATE INDEX IF NOT EXISTS roads_polygon_geom_idx ON {filename} USING GIST (geom);
    """
    print(cmd_indexes)


add_to_postgres("postgis", "yitong", "3_OHCA_with_geometric_pts")

ogr2ogr -f PostgreSQL PG:"dbname=postgis user=yitong" 3_OHCA_with_geometric_pts.gpkg -nln 3_OHCA_with_geometric_pts -lco GEOMETRY_NAME=geom -lco FID=id -nlt PROMOTE_TO_MULTI 3_OHCA_with_geometric_pts

CREATE INDEX IF NOT EXISTS hectare_grid_geom_idx ON hectare_grid USING GIST (geom);

CREATE INDEX IF NOT EXISTS roads_polygon_geom_idx ON 3_OHCA_with_geometric_pts USING GIST (geom);
    


#### intersect with hectare table

In [11]:
def intersect_ohca_with_hectares(output_table_name: str, input_table_name: str):
    cmd = f"""
DROP TABLE IF EXISTS {output_table_name};

CREATE TABLE {output_table_name} AS
SELECT 
    g.id AS grid_id,
    g.row_index AS grid_row,
    g.col_index AS grid_col,
    mp.id AS id,
    mp."case _" AS case_num,
    mp."date of incident" AS datetime,
    mp."location type" AS location_type,
    mp."location type other" AS location_type_other,
    mp."time call received at dispatch center" AS time,
    mp."lat" as lat,
    mp."lon" as lon,
    ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
FROM hectare_grid g
JOIN {'"'+input_table_name+'"'} mp
  ON ST_Intersects(g.geom, mp.geom);
"""

    print(cmd)

intersect_ohca_with_hectares("ohca_by_hectare", "3_ohca_with_geometric_pts")


DROP TABLE IF EXISTS ohca_by_hectare;

CREATE TABLE ohca_by_hectare AS
SELECT 
    g.id AS grid_id,
    g.row_index AS grid_row,
    g.col_index AS grid_col,
    mp.id AS id,
    mp."case _" AS case_num,
    mp."date of incident" AS datetime,
    mp."location type" AS location_type,
    mp."location type other" AS location_type_other,
    mp."time call received at dispatch center" AS time,
    mp."lat" as lat,
    mp."lon" as lon,
    ST_Intersection(ST_MakeValid(mp.geom), g.geom) AS geom
FROM hectare_grid g
JOIN "3_ohca_with_geometric_pts" mp
  ON ST_Intersects(g.geom, mp.geom);



#### Added a date column to the ohca_by_hectare table
ALTER TABLE ohca_by_hectare ADD date DATE;

#### Populate `date` column with the dates from the `datetime` column
UPDATE ohca_by_hectare SET date = DATE(datetime);

In [9]:
def extract_table_into_file(database_name: str, username: str, table_name: str):
    cmd = f"""
ogr2ogr -f GPKG {table_name}.gpkg PG:"dbname={database_name} user={username}" {table_name}
    """
    print(cmd)

In [10]:
extract_table_into_file("postgis", "yitong", "ohca_by_hectare")


ogr2ogr -f GPKG ohca_by_hectare.gpkg PG:"dbname=postgis user=yitong" ohca_by_hectare
    


#### convert the grid_classification file obtained from the using_openstreetmaps.ipynb notebook into csv so that it can be added to postgres

In [24]:
def convert_xlsx_to_csv(filepath: str):
    excel_filepath = filepath + "/grid_classification.xlsx"
    df = pd.read_excel(excel_filepath)
    csv_filepath = filepath + "/grid_classification.csv"
    df.to_csv(csv_filepath)
    print("Done")

In [32]:
filepath = BASE_DATASET_PATH / "geospatial_data/2018_geospatial"
filepath = str(filepath)
convert_xlsx_to_csv(filepath)

Done


#### Adding the csv file to postgres

In [33]:
def add_to_postgres(filename: str, year: str, dbname: str, username: str):
    cmd = f"""
    ogr2ogr -f "PostgreSQL" PG:"dbname={dbname} user={username}" {filename}.csv -nln {filename}_{year}
    """
    print(cmd)

add_to_postgres("grid_classification", "2018", "postgis", "yitong")


    ogr2ogr -f "PostgreSQL" PG:"dbname=postgis user=yitong" grid_classification.csv -nln grid_classification_2018
    


Typecast the columns so that they can be compared with the ohca_by_hectare table. Else the columns will be varchar

```
ALTER TABLE grid_classification_2018 
ALTER COLUMN grid_id TYPE integer USING grid_id::integer,
ALTER COLUMN grid_row TYPE bigint USING grid_row::bigint,
ALTER COLUMN grid_col TYPE bigint USING grid_col::bigint;
```

#### combine ohca incidence with classification of hectare grid in 2018

```DROP TABLE IF EXISTS ohca_by_hectare_2018;

CREATE TABLE ohca_by_hectare_2018 AS
SELECT
    g.grid_id AS grid_id,
    g.grid_row AS grid_row,
    g.grid_col AS grid_col,
    g.id AS id,
    g.date AS date,
    g.time AS time,
    g.location_type AS location_type,
    g.location_type_other AS location_type_other,
    c.hectare_classification AS hectare_classification
FROM
    ohca_by_hectare g
JOIN
    grid_classification_2018 c
ON
    g.grid_id = c.grid_id
WHERE
    EXTRACT(YEAR FROM g.date) = 2018;
```


#### Export out the table as geopackage

ogr2ogr -f CSV ohca_by_hectare_2018.csv PG:"dbname=postgis user=yitong" ohca_by_hectare_2018